# Variants from Vario output

The script contains steps where data are prepared but need to be prosessed by the shell scripts (vireo.s and vep.sh) to run. 

https://github.com/single-cell-genetics/cellsnp-lite/issues/109

In [ ]:
options(warn=-1)

In [ ]:
library_load <- suppressMessages(
    
    list(
        
        # Seurat 
        library(Seurat), 
        
        #  Rsamtools 
        library(Rsamtools), 
        
        # Data 
        library(GenomicRanges), 
        library(tidyverse), 
        library(data.table), 
        library(Matrix), 
        
        # Plotting 
        library(ggplot2), 
        library(patchwork), 
        library(ComplexHeatmap), 
        library(viridis), 
        
        # Pyhton compatibility
        library(reticulate)
        
    )
    
)

In [ ]:
# Configure reticulate 
use_condaenv(condaenv='p.3.8.12-FD20250213HSCT', conda="/nobackup/peer/fdeckert/miniconda3/bin/conda", required=NULL)
py_config()

In [ ]:
random_seed <- 42
set.seed(random_seed)

In [ ]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20250213HSCT")

In [ ]:
# Plotting Theme
source("plotting_global.R")
ggplot2::theme_set(theme_global_set(size_select=1)) # From project global source()

# Load Seurat object for meta data access 

In [ ]:
so <- readRDS("data/object/pp.rds")

# Load genotype

In [ ]:
# Genotype
genotype_file <- "result/genotype/vireo_mode_1a_genome1k.csv"
genotype <- read.csv(genotype_file, row.names=1) %>% dplyr::filter(minmaf=="cellsnp 0.0", donor_id %in% c("donor0", "donor1")) %>% dplyr::select(patient_id, donor_id, genotype_class) %>% distinct()
genotype <- reshape2::dcast(genotype, patient_id~donor_id, value.var="genotype_class") %>% dplyr::rename(PATIENT_ID=patient_id, GTC_DONOR0=donor0, GTC_DONOR1=donor1)

# Load reference annotation 

In [ ]:
gtf <- rtracklayer::import("../reference/genome/GRCh38/Homo_sapiens.GRCh38.110.gtf")

# GT variants import

In [ ]:
gt_import <- function(patient_id) {
    
    # Import GT donor file
    gt_vcf <- read.table(paste0("data/object/genotype/vireo/vireo_mode_1a_genome1k/", patient_id, "_vireo_cellsnp_0/GT_donors.vireo.vcf.gz"), header=FALSE)
    colnames(gt_vcf) <- c("CHROM", "POS", "ID", "REF", "ALT", "QUAL", "FILTER", "INFO", "FORMAT", "DONOR0", "DONOR1")
    gt_vcf$ID <- paste0(gt_vcf$CHROM, "_", gt_vcf$POS, "_", gt_vcf$REF, "_", gt_vcf$ALT)
    
    # Import variant tsv
    variant_vcf_file <- paste0("data/object/genotype/variants/vireo_mode_1a_genome1k/", patient_id, "_vireo_cellsnp_0/variants.tsv")
    variant_vcf <- read.table(variant_vcf_file, header=TRUE)
    colnames(variant_vcf) <- c("ID", "DONOR0", "DONOR1")
    
    # Filter GT donor by variants 
    gt_vcf$VARIANT <- gt_vcf$ID %in% variant_vcf$ID
    
    # Set variables 
    gt_vcf$GT_DONOR0 <- sapply(strsplit(gt_vcf$DONOR0, ":"), "[", 1) 
    gt_vcf$GT_DONOR1 <- sapply(strsplit(gt_vcf$DONOR1, ":"), "[", 1)
    gt_vcf$AD_DONOR0 <- as.numeric(sapply(strsplit(gt_vcf$DONOR0, ":"), "[", 2))
    gt_vcf$AD_DONOR1 <- as.numeric(sapply(strsplit(gt_vcf$DONOR1, ":"), "[", 2))
    gt_vcf$DP_DONOR0 <- as.numeric(sapply(strsplit(gt_vcf$DONOR0, ":"), "[", 3))
    gt_vcf$DP_DONOR1 <- as.numeric(sapply(strsplit(gt_vcf$DONOR1, ":"), "[", 3))
    
    gt_vcf$AD <- sapply(strsplit(sapply(strsplit(gt_vcf$INFO, ";"), "[", 1), "="), "[", 2) %>% as.numeric() 
    gt_vcf$DP <- sapply(strsplit(sapply(strsplit(gt_vcf$INFO, ";"), "[", 2), "="), "[", 2) %>% as.numeric() 
    gt_vcf$OTH <- sapply(strsplit(sapply(strsplit(gt_vcf$INFO, ";"), "[", 3), "="), "[", 2) %>% as.numeric()
    gt_vcf$CNT <- sum(gt_vcf$AD) + sum(gt_vcf$DP) + sum(gt_vcf$OTH)

    gt_vcf$PATIENT_ID <- patient_id
    
    return(gt_vcf)
    
}

In [ ]:
gt_vcf <- lapply(paste0("Patient_", 1:11), gt_import)
gt_vcf <- do.call("rbind", gt_vcf)

In [ ]:
# There are some duplicated entries - not sure why yet
gt_vcf <- distinct(gt_vcf)

In [ ]:
gt_vcf$PATIENT_ID <- factor(gt_vcf$PATIENT_ID, levels=paste0("Patient_", 1:11))

# Variant effect predictor 

In [ ]:
cache <- TRUE

In [ ]:
if(!cache) {
    
    for(PATIENT_ID in paste0("Patient_", 1:11)) {
    
        vep_input <- dplyr::filter(gt_vcf, PATIENT_ID==PATIENT_ID, OTH<AD, OTH<DP) %>% 
            dplyr::mutate(START=POS, END=POS, VARIANT=paste0(REF, "/", ALT), STRAND="+") %>% 
            dplyr::select(CHROM, START, END, VARIANT, STRAND) %>% 
            dplyr::arrange(CHROM, START) %>%
            unique() %>% as.data.frame() 

        vep_dir <- paste0("data/object/genotype/vep/", PATIENT_ID)
        vep_file <- paste0(vep_dir, "/gt_variants.tsv")
        if(!dir.exists(vep_dir)) {dir.create(vep_dir)}

        write.table(vep_input, vep_file, quote=FALSE, sep="\t", col.names=FALSE, row.names=FALSE)
    
    }   
    
}

In [ ]:
vep_import <- function(PATIENT_ID) {
    
    vep <- read.table(paste0("data/object/genotype/vep/", PATIENT_ID, "/variant_effect_output.txt"))
    colnames(vep) <- toupper(c("id", "Location", "Allele", "Gene", "Feature", "Feature_type", "Consequence", "cDNA_position", "CDS_position", "Protein_position", "Amino_acids", "Codons", "Existing_variation", "Extra"))
    
    # Annotate 
    ref <- as.data.frame(gtf)[, c("gene_id", "gene_name")] %>% na.omit() %>% unique()
    colnames(ref) <- c("GENE", "SYMBOL")
    
    vep <- dplyr::left_join(vep, ref, by=join_by(GENE))
    
    # Fix variant ID
    vep$ID <- gsub("/", "_", vep$ID)
    
    vep <- vep[, c("ID", "ALLELE", "FEATURE_TYPE", "CONSEQUENCE", "SYMBOL")]
    
    # Set patient id
    vep$PATIENT_ID <- PATIENT_ID
    
    return(vep)
}

In [ ]:
vep <- lapply(paste0("Patient_", 1:11), vep_import)
vep <- do.call("rbind", vep)

In [ ]:
# There are some duplicated entries - not sure why yet
vep <- distinct(vep)

# Combine gt variants and vep 

In [ ]:
data <- dplyr::left_join(gt_vcf, vep, by=join_by(ID, PATIENT_ID))

# Vireo manual genotype annotation 

In [ ]:
data <- dplyr::left_join(data, genotype, by=join_by(PATIENT_ID))

# Add aligned read count, percentage cells expressing a gene and selected meta data

In [ ]:
meta <- dplyr::select(so@meta.data, sample_path, patient_id, gvhd_class) %>% dplyr::distinct()

In [ ]:
bam_reads_aligned <- lapply(meta$sample_path, function(sample_path) {
    
    # Get alignment stats
    idxstats <- idxstatsBam(paste0(sample_path, "/possorted_genome_bam.bam"))

    # Sum the number of aligned reads
    total_aligned_reads <- sum(idxstats$mapped)
    
    return(total_aligned_reads)

}
                          )

In [ ]:
meta$bam_reads_aligned <- unlist(bam_reads_aligned)

In [ ]:
mtx <- lapply(meta$sample_path, function(sample_path) {
    
    # Load data matrix
    mtx <- Seurat::ReadMtx(
        
        mtx=paste0(sample_path, "/filtered_feature_bc_matrix/matrix.mtx.gz"), 
        features=paste0(sample_path, "/filtered_feature_bc_matrix/features.tsv.gz"),
        cells=paste0(sample_path, "/filtered_feature_bc_matrix/barcodes.tsv.gz")
        
    )
    
    # make unique cell names
    idx <- strsplit(rownames(meta[meta$sample_path==sample_path, , drop=FALSE]), "_")[[1]][2]
    colnames(mtx) <- paste0(colnames(mtx), "_", idx)
    
    return(mtx)
    
}
                           )

In [ ]:
mtx <- do.call("cbind", mtx)

In [ ]:
cpm <- mtx %*% Matrix::Diagonal(x=1 / Matrix::colSums(mtx))
cpm <- 10^6 * cpm

In [ ]:
# Add patient id based information 
colnames(meta) <- toupper(colnames(meta))
data <- dplyr::left_join(data, dplyr::group_by(meta, PATIENT_ID, GVHD_CLASS) %>% dplyr::summarise(BAM_READS_ALIGNED=sum(BAM_READS_ALIGNED)) %>% dplyr::ungroup(), by=join_by(PATIENT_ID))

In [49]:
# Add gene level annotation 
data <- dplyr::left_join(data, data.frame(PCT=round(rowSums(mtx>0)/ncol(mtx), 5), CPM=rowMeans(cpm)) %>% tibble::rownames_to_column(., "SYMBOL"), by=join_by(SYMBOL))

# Save final output 

In [50]:
write.csv(data, "result/genotype/gt_vep.csv")